# Open-dLLM Phase 4: Modern Block Diffusion Training

Train a modern block diffusion LLM on multi-source 100B dataset using Kaggle 2xT4 GPUs.

- **Model**: ~213M params (16L/1024d/16h/4kv/2816MLP, tied embeddings), seq_len=1024, block_size=32
- **Data**: FinePDFs 50B + DCLM 30B + FineWeb-Edu 20B (pre-shuffled, streaming)
- **Key innovations**: FlexAttention staircase mask, GQA 4:1, Liger fused kernels,
  AMP fp16, gradient checkpointing, CART noise rescheduling, complementary masking,
  WSD scheduler, Muon optimizer, MLP dropout 0.1, torch.compile
- **Hardware**: 2xT4 (CC 7.5, 16GB each) with DDP multi-GPU training

In [ ]:
# Install deps — liger-kernel requires CC 7.0+ (T4 = CC 7.5)
# muon: KellerJordan's optimizer (NOT the bioinformatics 'muon' package on PyPI)
!pip install -q torch datasets tokenizers liger-kernel
!pip uninstall -y muon 2>/dev/null; pip install -q git+https://github.com/KellerJordan/Muon

In [ ]:
!git clone https://github.com/hahuyhoang411/Open-dLLM.git
%cd Open-dLLM

In [ ]:
# Retrain BPE tokenizer with correct special tokens on new dataset
# Downloads 100K docs (streaming), trains BPE — ~3-5 min on Kaggle
!python 04_modern_dllm/train_tokenizer.py --force

In [ ]:
import torch
n_gpus = torch.cuda.device_count()
print(f"GPUs: {n_gpus}")
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    cc = torch.cuda.get_device_capability(i)
    print(f"  [{i}] {name} — {vram:.1f} GB — CC {cc[0]}.{cc[1]}")

print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

cc = torch.cuda.get_device_capability(0)
if cc >= (7, 0):
    print(f"\nCC {cc[0]}.{cc[1]} — Liger Kernel + FlexAttention + torch.compile compatible")
else:
    print(f"\nCC {cc[0]}.{cc[1]} — will use fallbacks (no Liger/FlexAttention/compile)")

In [ ]:
# Train with DDP on 2xT4: torchrun handles multi-GPU setup
# --no-compile for DDP: torch.compile + Triton hits T4 shared memory limits
# Liger fused kernels already handle the hot path (SwiGLU, CrossEntropy, RMSNorm)
# VRAM: ~6-8GB per T4 (Liger fused kernels)
import torch
n_gpus = torch.cuda.device_count()

if n_gpus > 1:
    # DDP: grad_accum=2 keeps same effective batch as single-GPU accum=4
    !torchrun --nproc_per_node={n_gpus} 04_modern_dllm/modern_dllm.py \
        --train --seq-len 1024 --block-size 32 \
        --batch-size 16 --grad-accum-steps 2 --no-compile
else:
    # Single GPU: torch.compile works fine without DDP
    !python 04_modern_dllm/modern_dllm.py --train --seq-len 1024 --block-size 32 \
        --batch-size 16 --grad-accum-steps 4

In [ ]:
# Generate sample text
!python 04_modern_dllm/modern_dllm.py --seq-len 1024 --block-size 32 \
    --prompt "The meaning of life is" --max-tokens 128 --denoise-steps 10

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/weights', exist_ok=True)
src = '04_modern_dllm/weights/modern_dllm_b32.pt'
if os.path.exists(src):
    shutil.copy(src, '/kaggle/working/weights/')
    size_mb = os.path.getsize(f'/kaggle/working/weights/{os.path.basename(src)}') / 1e6
    print(f"Saved weights to /kaggle/working/weights/ ({size_mb:.1f} MB)")
else:
    print(f"Weights not found at {src}")